# Final prediction for all four tasks

This notebook assembles the final classification submission in the issued template order and validates the Task 4 retrieval artefacts. Tasks 1--3 own their model-specific inference; this notebook consumes their prediction CSVs, rejects stale or misaligned rows, and writes one combined file only when every required column is complete. Task 2 is intentionally isolated in one configuration entry so its final artefact can be swapped without changing the merge logic.


## 1. Configuration

Set `WRITE_FINAL = True` only for the last submission run. Until then, the notebook performs a preflight and writes a clearly named draft only when all classification predictions are available.

The submission filename follows the convention in section 5 of the brief, `COSC2753_A2_<Group number>_<studentID1_studentID2_…>`, built from `GROUP_ID` and `STUDENT_IDS` so it cannot drift: *"If your submission does not follow the name convention, the mark deduction will be applied."*

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), 'Run this notebook from the repository root or notebooks/'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_paths

GROUP_ID = 'SG_G3'
STUDENT_IDS = ['s4040502', 's4034066', 's4030327', 's4054071', 's3974820']
SUBMISSION_NAME = '_'.join(['COSC2753_A2', GROUP_ID, *STUDENT_IDS])
WRITE_FINAL = True
RUN_TASK4_TEST_RETRIEVAL = False
TASK4_TOP_K = 10

RUN_MODELS = True
TASK_CHECKPOINTS = {
    'articleType': PROJECT_ROOT / 'artifacts/task1/submitted',                         # a directory: deployment.json names the .pt
    'season': PROJECT_ROOT / 'artifacts/task2/submitted/task2_model.pt',              # DenseNet-121
    'gender_usage': PROJECT_ROOT / 'artifacts/task3/submitted/task3_gender_usage_C_weighted.pt',  # design C
    'retrieval': PROJECT_ROOT / 'artifacts/task4/submitted/arcface_best.pt',          # ArcFace encoder
}

PREDICTION_SOURCES = {
    'articleType': [PROJECT_ROOT / 'predictions/task1/task1_predictions.csv'],
    'season': [PROJECT_ROOT / 'predictions/task2/task2_predictions.csv'],
    'gender_usage': [PROJECT_ROOT / 'predictions/task3/task3_gender_usage_nguyen.csv'],
}

FINAL_DIR = PROJECT_ROOT / 'predictions/final'
FINAL_PATH = FINAL_DIR / f'{SUBMISSION_NAME}.csv'
DRAFT_PATH = FINAL_DIR / f'DRAFT_{SUBMISSION_NAME}.csv'
print('Project:', PROJECT_ROOT)
print('Final output:', FINAL_PATH)

Project: D:\Dowloads\Machine-Learning-Assignment-2
Final output: D:\Dowloads\Machine-Learning-Assignment-2\predictions\final\COSC2753_A2_SG_G3_s4040502_s4034066_s4030327_s4054071_s3974820.csv


## 2. Resolve the issued template and test images

The template is the authority for row count, ID values, row order, and column order. The resolver supports the repository layout and the two local staging layouts used by the task notebooks.


In [2]:
EXPECTED_COLUMNS = ['id', 'gender', 'articleType', 'season', 'usage']
TEMPLATE_CANDIDATES = [
    data_paths.test_template(),
    PROJECT_ROOT / 'predictions/task1/task1_predictions.csv',
]
TEST_IMAGE_DIR_CANDIDATES = [data_paths.test_images()]
TEMPLATE_PATH = next((p.resolve() for p in TEMPLATE_CANDIDATES if p.is_file()), None)
assert TEMPLATE_PATH is not None, 'No issued template or Task 1 template-preserving prediction CSV was found'
TEST_IMAGE_DIR = next((p.resolve() for p in TEST_IMAGE_DIR_CANDIDATES if p.is_dir()), None)
template = pd.read_csv(TEMPLATE_PATH, dtype={'id': 'int64'})
assert list(template.columns) == EXPECTED_COLUMNS, (template.columns.tolist(), EXPECTED_COLUMNS)
assert template['id'].is_unique and template['id'].notna().all()
print(f'Template: {TEMPLATE_PATH} ({len(template):,} rows)')
print(f'Test images: {TEST_IMAGE_DIR or "NOT STAGED (classification merge can still run)"}')

Template: D:\Dowloads\A2 Machine Learning\Dataset\FashionDataset\test\styles_prediction.csv (5,829 rows)
Test images: D:\Dowloads\A2 Machine Learning\Dataset\FashionDataset\test\images_test


## 3. Run the four saved models

Each task loads its own checkpoint from `artifacts/` and writes its prediction CSV. Task 4 is a retrieval system with no column in the issued template, so it is preflighted and run separately in sections 5 and 6.

Running them is the point. A prediction CSV sitting in the repository proves only that somebody once exported one — Task 1's submission copy was stale by 709 of 5,829 rows for exactly that reason. Each model carries its own class order, image size and normalisation statistics, and inference reads all of it rather than re-deriving anything: a statistic recomputed on the test images would be fitted to the data being predicted. A task whose checkpoint is absent is reported, not fatal — the merge falls back to the CSV already on disk.

In [3]:
import traceback

run_log = []
if RUN_MODELS:
    for task, runner in [
        ('articleType', 'src.task1.task1_inference'),
        ('season', 'src.task2_utils'),
        ('gender_usage', 'src.task3.predict_test'),
    ]:
        checkpoint = TASK_CHECKPOINTS[task]
        target = PREDICTION_SOURCES[task][0]
        exists = checkpoint.exists()
        try:
            if not exists:
                raise FileNotFoundError(f'{checkpoint} is absent')
            if task == 'articleType':
                from src.task1.task1_inference import Predictor, predict_template
                predictor = Predictor(checkpoint)
                written = predict_template(predictor, TEMPLATE_PATH, TEST_IMAGE_DIR, target)
                detail = f'{predictor.arm}, {written:,} rows'
            elif task == 'season':
                from src.task2_utils import predict_test_set
                predict_test_set(checkpoint_path=checkpoint, output_path=target)
                detail = 'DenseNet-121'
            else:
                import subprocess
                done = subprocess.run(
                    [sys.executable, str(PROJECT_ROOT / 'src/task3/predict_test.py'),
                     '--checkpoint', str(checkpoint), '--output', str(target)],
                    capture_output=True, text=True, cwd=PROJECT_ROOT)
                if done.returncode:
                    raise RuntimeError(done.stderr.strip().splitlines()[-1])
                detail = 'design C + mirror TTA'
            status = 'RAN'
        except Exception as exc:
            detail = f'{type(exc).__name__}: {exc}'
            status = 'SKIPPED' if target.is_file() else 'FAILED'
        run_log.append({'task': task, 'checkpoint': checkpoint.name, 'status': status,
                        'detail': detail[:90]})
else:
    run_log.append({'task': '(all)', 'checkpoint': '-', 'status': 'OFF',
                    'detail': 'RUN_MODELS is False; merging the CSVs already on disk'})
display(pd.DataFrame(run_log))

,task,checkpoint,status,detail
0,articleType,submitted,RAN,"resnet_resample, 5,829 rows"
1,season,task2_model.pt,SKIPPED,ModuleNotFoundError: No module named 'skimage'
2,gender_usage,task3_gender_usage_C_weighted.pt,RAN,design C + mirror TTA


## 4. Load and align Task 1--3 predictions

Each source must contain `id` plus its owned output column(s). Rows are joined by ID and then restored to template order; positional concatenation is never used. Blank labels, duplicate IDs, missing IDs, extra IDs, and conflicting non-owned columns fail loudly.


In [4]:
TASK_COLUMNS = {
    'articleType': ['articleType'],
    'season': ['season'],
    'gender_usage': ['gender', 'usage'],
}

def first_existing(candidates: list[Path]) -> Path | None:
    return next((p.resolve() for p in candidates if p.is_file()), None)

def load_task_predictions(path: Path, owned_columns: list[str], expected_ids: pd.Series) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype={'id': 'int64'})
    required = ['id', *owned_columns]
    missing_columns = [column for column in required if column not in frame.columns]
    assert not missing_columns, f'{path}: missing columns {missing_columns}'
    assert frame['id'].notna().all() and frame['id'].is_unique, f'{path}: IDs must be unique and non-null'
    for column in owned_columns:
        values = frame[column].astype('string').str.strip()
        assert values.notna().all() and values.ne('').all(), f'{path}: blank values in {column}'
        frame[column] = values
    expected = set(expected_ids.astype(int))
    actual = set(frame['id'].astype(int))
    assert actual == expected, (
        f'{path}: ID mismatch; missing={sorted(expected - actual)[:10]}, '
        f'extra={sorted(actual - expected)[:10]}'
    )
    return frame[required]

submission = template.copy()
source_records = []
missing_tasks = []
for task_name, owned_columns in TASK_COLUMNS.items():
    source_path = first_existing(PREDICTION_SOURCES[task_name])
    if source_path is None:
        missing_tasks.append(task_name)
        source_records.append({'task': task_name, 'status': 'MISSING', 'path': None})
        continue
    task_frame = load_task_predictions(source_path, owned_columns, template['id'])
    aligned = template[['id']].merge(task_frame, on='id', how='left', validate='one_to_one')
    for column in owned_columns:
        submission[column] = aligned[column]
    source_records.append({'task': task_name, 'status': 'READY', 'path': str(source_path)})

source_status = pd.DataFrame(source_records)
display(source_status)
display(submission.head())

,task,status,path
0,articleType,READY,D:\Dowloads\Machine-Learning-Assignment-2\pred...
1,season,READY,D:\Dowloads\Machine-Learning-Assignment-2\pred...
2,gender_usage,READY,D:\Dowloads\Machine-Learning-Assignment-2\pred...


,id,gender,articleType,season,usage
0,52003,Men,Watches,Fall,Casual
1,52007,Women,Dresses,Summer,Casual
2,52017,Men,Watches,Summer,Casual
3,52021,Women,Lounge Pants,Summer,Casual
4,52023,Men,Tshirts,Summer,Casual


## 5. Validate classification labels and write the combined CSV

Known label vocabularies are derived from the cleaned training manifest when it is available. The final file is blocked until Task 1, Task 2, and Task 3 are all present.


In [5]:
TRAIN_MANIFEST_CANDIDATES = [data_paths.train_table()]
train_manifest_path = first_existing(TRAIN_MANIFEST_CANDIDATES)
label_audit = []
if train_manifest_path is not None:
    train_manifest = pd.read_csv(train_manifest_path, low_memory=False)
    for column in EXPECTED_COLUMNS[1:]:
        if column not in submission or submission[column].isna().any() or column not in train_manifest:
            continue
        known = set(train_manifest[column].dropna().astype(str).str.strip())
        predicted = set(submission[column].astype(str).str.strip())
        unknown = sorted(predicted - known)
        label_audit.append({'column': column, 'predicted_classes': len(predicted), 'unknown': unknown})
        assert not unknown, f'{column}: labels absent from training vocabulary: {unknown}'
display(pd.DataFrame(label_audit))

FINAL_DIR.mkdir(parents=True, exist_ok=True)
classification_ready = not missing_tasks
if classification_ready:
    assert list(submission.columns) == EXPECTED_COLUMNS
    assert submission['id'].equals(template['id']), 'Template row order changed'
    assert submission[EXPECTED_COLUMNS[1:]].notna().all().all(), 'Final labels contain nulls'
    assert submission[EXPECTED_COLUMNS[1:]].apply(lambda s: s.astype(str).str.strip().ne('').all()).all()
    output_path = FINAL_PATH if WRITE_FINAL else DRAFT_PATH
    submission.to_csv(output_path, index=False, lineterminator='\n')
    print(f'Wrote {len(submission):,} rows -> {output_path}')
else:
    output_path = None
    print('Combined CSV not written. Missing task artefacts:', ', '.join(missing_tasks))

,column,predicted_classes,unknown
0,gender,5,[]
1,articleType,95,[]
2,season,4,[]
3,usage,5,[]


Wrote 5,829 rows -> D:\Dowloads\Machine-Learning-Assignment-2\predictions\final\COSC2753_A2_SG_G3_s4040502_s4034066_s4030327_s4054071_s3974820.csv


## 6. Task 4 retrieval artefact preflight

Task 4 is a retrieval system and therefore has no column in `styles_prediction.csv`. Its submission contract is the frozen ArcFace encoder plus a gallery embedding matrix whose rows align exactly with the gallery IDs. The optional next section can encode every test image and save its Top-K catalogue neighbours.

The gallery comes from `artifacts/task4/submitted/`, not from the folder holding the training checkpoint. That one is an earlier development population of 30,389 items against the 33,968 the hold-out benchmark scored; retrieving against it would answer queries from a catalogue missing 3,579 items, and would not be the system the mAP@10 figure describes.

In [6]:
TASK4_DIR = PROJECT_ROOT / 'artifacts/task4/submitted'
TASK4_EMBEDDING_DIR = TASK4_DIR
TASK4_FILES = {
    'checkpoint': TASK4_DIR / 'arcface_best.pt',
    'gallery_embeddings': TASK4_EMBEDDING_DIR / 'gallery_embeddings.npy',
    'gallery_ids': TASK4_EMBEDDING_DIR / 'gallery_ids.npy',
    'preprocessing': TASK4_DIR / 'image_preprocessing.json',
}
task4_status = pd.DataFrame([
    {'artefact': name, 'ready': path.is_file(), 'path': str(path)}
    for name, path in TASK4_FILES.items()
])
display(task4_status)
task4_ready = bool(task4_status['ready'].all())
if task4_ready:
    gallery_ids = np.load(TASK4_FILES['gallery_ids'], mmap_mode='r')
    gallery_embeddings = np.load(TASK4_FILES['gallery_embeddings'], mmap_mode='r')
    assert gallery_ids.ndim == 1 and gallery_embeddings.ndim == 2
    assert len(gallery_ids) == len(gallery_embeddings), 'Task 4 gallery IDs/embeddings are misaligned'
    assert len(np.unique(gallery_ids)) == len(gallery_ids), 'Task 4 gallery IDs are not unique'
    norms = np.linalg.norm(np.asarray(gallery_embeddings[: min(2048, len(gallery_embeddings))]), axis=1)
    assert np.allclose(norms, 1.0, atol=2e-3), 'Task 4 gallery embeddings are not L2-normalised'
    print(f'Task 4 ready: {len(gallery_ids):,} gallery items, {gallery_embeddings.shape[1]} dimensions')
else:
    print('Task 4 retrieval export is blocked until every required artefact exists.')

,artefact,ready,path
0,checkpoint,True,D:\Dowloads\Machine-Learning-Assignment-2\arti...
1,gallery_embeddings,True,D:\Dowloads\Machine-Learning-Assignment-2\arti...
2,gallery_ids,True,D:\Dowloads\Machine-Learning-Assignment-2\arti...
3,preprocessing,True,D:\Dowloads\Machine-Learning-Assignment-2\arti...


Task 4 ready: 33,968 gallery items, 512 dimensions


## 7. Optional Task 4 Top-K predictions for all test queries

This section is off during ordinary preflight. When enabled, it restores the ArcFace encoder, applies the recorded letterbox and normalization, retrieves by cosine similarity, and saves one row per `(query, rank)` pair. It does not alter the classification submission.


In [7]:
if RUN_TASK4_TEST_RETRIEVAL:
    assert task4_ready, 'Task 4 artefacts failed preflight'
    import torch
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
    from src.task4.models import ResNet18Encoder

    with TASK4_FILES['preprocessing'].open(encoding='utf-8') as handle:
        prep = json.load(handle)
    size = tuple(prep['resize']['target_size'])
    mean = prep['normalization']['mean_rgb']
    std = prep['normalization']['std_rgb']
    pad = tuple(prep['resize']['padding_color_rgb'])

    class Letterbox:
        def __init__(self, target_size, fill):
            self.height, self.width = target_size
            self.fill = fill
        def __call__(self, image):
            image = image.convert('RGB')
            scale = min(self.width / image.width, self.height / image.height)
            resized = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.BILINEAR)
            canvas = Image.new('RGB', (self.width, self.height), self.fill)
            canvas.paste(resized, ((self.width - resized.width) // 2, (self.height - resized.height) // 2))
            return canvas

    transform = transforms.Compose([Letterbox(size, pad), transforms.ToTensor(), transforms.Normalize(mean, std)])

    class TestImages(Dataset):
        def __init__(self, ids): self.ids = [int(i) for i in ids]
        def __len__(self): return len(self.ids)
        def __getitem__(self, index):
            image_id = self.ids[index]
            with Image.open(TEST_IMAGE_DIR / f'{image_id}.jpg') as image:
                tensor = transform(image)
            return image_id, tensor

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(TASK4_FILES['checkpoint'], map_location='cpu', weights_only=False)
    state = checkpoint.get('model_state_dict', checkpoint)
    encoder_state = {k: v for k, v in state.items() if not k.startswith('arcface_loss.')}
    model = ResNet18Encoder()
    model.load_state_dict(encoder_state, strict=True)
    model.to(device).eval()

    assert TEST_IMAGE_DIR is not None, 'Stage images_test before running Task 4 retrieval'
    missing_images = [int(i) for i in template['id'] if not (TEST_IMAGE_DIR / f'{i}.jpg').is_file()]
    assert not missing_images, f'Missing {len(missing_images)} test images; first IDs: {missing_images[:10]}'
    query_ids, query_vectors = [], []
    loader = DataLoader(TestImages(template['id']), batch_size=256, shuffle=False, num_workers=0, pin_memory=device.type == 'cuda')
    with torch.inference_mode():
        for ids, images in loader:
            query_ids.extend(ids.numpy().astype(int).tolist())
            query_vectors.append(model(images.to(device, non_blocking=True)).cpu().numpy())
    query_vectors = np.concatenate(query_vectors).astype('float32')
    gallery_matrix = np.asarray(gallery_embeddings, dtype='float32')

    rows = []
    block_size = 256
    k = min(TASK4_TOP_K, len(gallery_ids))
    for start in range(0, len(query_vectors), block_size):
        scores = query_vectors[start:start + block_size] @ gallery_matrix.T
        top = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
        top_scores = np.take_along_axis(scores, top, axis=1)
        order = np.argsort(-top_scores, axis=1)
        top = np.take_along_axis(top, order, axis=1)
        top_scores = np.take_along_axis(top_scores, order, axis=1)
        for offset, query_id in enumerate(query_ids[start:start + block_size]):
            for rank, (gallery_index, score) in enumerate(zip(top[offset], top_scores[offset]), start=1):
                rows.append({'query_id': query_id, 'rank': rank, 'gallery_id': int(gallery_ids[gallery_index]), 'cosine_similarity': float(score)})
    retrieval = pd.DataFrame(rows)
    retrieval_path = FINAL_DIR / f'COSC2753_A2_{GROUP_ID}_task4_top{TASK4_TOP_K}.csv'
    retrieval.to_csv(retrieval_path, index=False, lineterminator='\n')
    print(f'Wrote {len(retrieval):,} retrieval rows -> {retrieval_path}')
    display(retrieval.head(2 * TASK4_TOP_K))
else:
    print('Task 4 test retrieval skipped. Set RUN_TASK4_TEST_RETRIEVAL = True for the final artefact run.')

Task 4 test retrieval skipped. Set RUN_TASK4_TEST_RETRIEVAL = True for the final artefact run.


## 8. Final manifest and submission gate

The manifest records the exact input files and hashes used by the final run. It is written only when the combined classification CSV exists; Task 4 readiness is recorded separately because retrieval is not part of the issued CSV schema.


In [8]:
def sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

if output_path is not None:
    reloaded = pd.read_csv(output_path, dtype={'id': 'int64'})
    assert list(reloaded.columns) == EXPECTED_COLUMNS
    assert reloaded['id'].equals(template['id'])
    assert reloaded[EXPECTED_COLUMNS[1:]].notna().all().all()
    manifest = {
        'group_id': GROUP_ID,
        'classification_file': str(output_path.relative_to(PROJECT_ROOT)),
        'classification_sha256': sha256(output_path),
        'template': str(TEMPLATE_PATH),
        'template_sha256': sha256(TEMPLATE_PATH),
        'rows': len(reloaded),
        'columns': EXPECTED_COLUMNS,
        'sources': source_records,
        'task4_ready': task4_ready,
        'task4_files': {name: {'path': str(path), 'sha256': sha256(path) if path.is_file() else None} for name, path in TASK4_FILES.items()},
    }
    manifest_path = output_path.with_suffix('.manifest.json')
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Manifest:', manifest_path)
    print('Classification SHA-256:', manifest['classification_sha256'])
    if WRITE_FINAL:
        print('FINAL SUBMISSION READY')
    else:
        print('DRAFT ONLY: inspect results, then set WRITE_FINAL = True and Run All.')
else:
    print('NOT READY:', {'missing_classification_tasks': missing_tasks, 'task4_ready': task4_ready})

Manifest: D:\Dowloads\Machine-Learning-Assignment-2\predictions\final\COSC2753_A2_SG_G3_s4040502_s4034066_s4030327_s4054071_s3974820.manifest.json
Classification SHA-256: 3b6b69308bcd8c0aeca3726b89fc2629d5725fed0b01e7ff3a0a898fd0623eee
FINAL SUBMISSION READY
